# 第四章：Memory（对话历史与状态管理）

## 学习目标

- 理解“无状态”模型为什么需要外部记忆管理
- 手动维护消息列表实现多轮对话
- 使用 `ChatMessageHistory` 存储对话记录
- 使用 `RunnableWithMessageHistory` 自动注入历史
- 用 `trim_messages` 控制上下文窗口长度
- 实现摘要记忆压缩长对话

## 0. 环境准备

In [96]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import langchain
print(f"langchain 版本: {langchain.__version__}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)
print("模型初始化完成")

langchain 版本: 1.2.13
模型初始化完成


## 1. 对比：无记忆 vs 有记忆

先直观感受一下问题——模型真的不记事。

In [97]:
# 第一轮：告诉模型你的名字
response1 = llm.invoke("我叫小明，请记住我的名字。")
print(f"第一轮回复: {response1.content}")

第一轮回复: 好的，小明，我已经记住你的名字了。很高兴认识你！😊


In [98]:
# 第二轮：问它你叫什么
response2 = llm.invoke("我叫什么名字？")
print(f"第二轮回复: {response2.content}")

第二轮回复: 您没有告诉我您的名字，所以我无法知道您叫什么。

如果您愿意分享的话，我很乐意知道该怎么称呼您！


模型无法回答——因为第二次调用和第一次完全无关。每次 `invoke` 都是独立的 HTTP 请求，模型不知道之前发生了什么。

In [99]:
# 把两轮对话放在同一个消息列表中，模型就“记住”了
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="我叫小明，请记住我的名字。"),
    AIMessage(content="好的，小明！我记住了你的名字。"),
    HumanMessage(content="我叫什么名字？"),
]

response = llm.invoke(messages)
print(f"回复: {response.content}")

回复: 你叫小明。


### 刚才发生了什么？

LLM 本身是**无状态**的。所谓的“记忆”，就是把之前的对话消息一起发给模型。模型看到了完整的上下文，自然就能“记住”之前说过的话。

所以 Memory 要解决的核心问题只有一个：**谁来管理这个消息列表？**

| 方式 | 做法 |
|------|------|
| 不管理 | 每次调用都是独立的，模型“失忆” |
| 手动管理 | 自己维护一个 Python list，每轮对话追加消息 |
| 框架管理 | 让 LangChain 自动加载、保存历史记录 |

## 2. 手动管理消息列表

第二章中我们用过 `MessagesPlaceholder` 把历史消息注入模板。现在把它做成一个完整的多轮对话。

In [100]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手。"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

In [101]:
history = []

# 第一轮
response = chain.invoke({"history": history, "input": "我叫小明，我是一名程序员"})
print(f"助手: {response}")
history.append(HumanMessage(content="我叫小明，我是一名程序员"))
history.append(AIMessage(content=response))

助手: 你好小明！很高兴认识你 👋

作为一名程序员，你平时主要用什么编程语言和技术栈呢？是前端、后端、全栈，还是专注于某个特定领域（比如AI、嵌入式、游戏开发等）？

有什么我可以帮你的吗？无论是技术问题、代码调试，还是闲聊都可以 😊


In [102]:
# 第二轮
response = chain.invoke({"history": history, "input": "我叫什么名字？我的职业是什么？"})
print(f"助手: {response}")
history.append(HumanMessage(content="我叫什么名字？我的职业是什么？"))
history.append(AIMessage(content=response))

助手: 你叫**小明**，职业是**程序员**。

我记住啦！😊


In [103]:
# 第三轮
response = chain.invoke({"history": history, "input": "给我推荐一本适合我职业的书"})
print(f"助手: {response}")

助手: 作为程序员，推荐一本经典之作：

**《代码整洁之道》（Clean Code）— Robert C. Martin**

这本书讲的是如何写出**可读、可维护、优雅**的代码，是程序员进阶的必修课。无论你是哪种技术栈，"Clean Code" 的理念都适用。

---

如果你想要其他方向的推荐，也可以告诉我：
- 系统架构 → 《设计模式》或《架构整洁之道》
- 算法基础 → 《算法导论》或《编程珠玑》
- 工程实践 → 《程序员修炼之道》
- 特定语言（比如 Python/Go/JavaScript）的进阶书

你目前对哪方面更感兴趣？


### 刚才发生了什么？

多轮对话正常工作了——模型记住了名字和职业。但这段代码有明显的痛点：

1. **手动 append**：每轮对话后都要手动把 Human 和 AI 消息加到 `history` 列表里
2. **容易出错**：忘了 append 或 append 顺序错了，对话就乱了
3. **重复代码**：每次调用的模式一模一样，应该抽象出来

接下来让 LangChain 帮我们做这件事。

## 3. ChatMessageHistory（消息存储）

在让框架自动管理之前，先认识底层的消息存储组件：`ChatMessageHistory`。

In [104]:
from langchain_community.chat_message_histories import ChatMessageHistory

chat_history = ChatMessageHistory()

# 添加消息
chat_history.add_user_message("我叫小明")
chat_history.add_ai_message("你好小明！很高兴认识你。")
chat_history.add_user_message("我叫什么？")

# 查看所有消息
for msg in chat_history.messages:
    print(f"{msg.type}: {msg.content}")

human: 我叫小明
ai: 你好小明！很高兴认识你。
human: 我叫什么？


In [105]:
# 清空历史
chat_history.clear()
print(f"清空后消息数: {len(chat_history.messages)}")

清空后消息数: 0


### ChatMessageHistory 关键 API

| 方法/属性 | 说明 |
|-----------|------|
| `add_user_message(content)` | 添加一条用户消息 |
| `add_ai_message(content)` | 添加一条 AI 消息 |
| `add_message(message)` | 添加任意类型的 Message 对象 |
| `messages` | 返回所有消息的列表 |
| `clear()` | 清空所有消息 |

这个类本身只是一个**内存中的列表包装器**，比直接用 `list` 好在：
- 接口统一，便于替换为数据库、Redis 等持久化后端
- 自动创建正确的 Message 类型

## 4. RunnableWithMessageHistory（自动管理历史）

这是本章的核心组件。它包装一个链，自动完成三件事：
1. 调用前：从存储中加载历史消息
2. 调用时：把历史注入到链的输入中
3. 调用后：把新的用户消息和 AI 回复保存到存储中

In [106]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# 用字典模拟多用户的会话存储
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """根据 session_id 获取（或创建）对话历史"""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 包装之前定义的 chain
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",       # 输入中哪个 key 是用户消息
    history_messages_key="history",    # 模板中哪个占位符放历史
)

print("chain_with_history 创建完成")

chain_with_history 创建完成


In [107]:
# 通过 config 指定 session_id
config = {"configurable": {"session_id": "user_001"}}

# 第一轮：告诉它你的信息
response = chain_with_history.invoke(
    {"input": "我叫小明，今年 25 岁，住在北京"},
    config=config,
)
print(f"助手: {response}")

助手: 你好小明！很高兴认识你 👋

25岁，住在北京——那是个很棒的年纪和城市！你是在北京工作、学习，还是其他方面呢？有什么我可以帮你的吗？


In [108]:
# 第二轮：验证它记住了
response = chain_with_history.invoke(
    {"input": "我叫什么？住在哪里？"},
    config=config,
)
print(f"助手: {response}")

助手: 你叫**小明**，今年25岁，住在**北京**。😊


In [109]:
# 查看存储中保存的完整历史
print("=== user_001 的对话历史 ===")
for msg in store["user_001"].messages:
    label = msg.type
    content = msg.content[:50] + "..." if len(msg.content) > 50 else msg.content
    print(f"{label}: {content}")

=== user_001 的对话历史 ===
human: 我叫小明，今年 25 岁，住在北京
ai: 你好小明！很高兴认识你 👋

25岁，住在北京——那是个很棒的年纪和城市！你是在北京工作、学习，还是...
human: 我叫什么？住在哪里？
ai: 你叫**小明**，今年25岁，住在**北京**。😊


### 刚才发生了什么？

对比手动管理：

| 手动方式 | RunnableWithMessageHistory |
|----------|---------------------------|
| 自己创建 `history = []` | 自动通过 `get_session_history` 获取 |
| 每轮手动 `history.append(...)` | 自动保存用户消息和 AI 回复 |
| 每次 invoke 传 `{"history": history}` | 自动注入，只需传 `{"input": ...}` |

我们只需要关心两件事：
1. **怎么存**——实现 `get_session_history` 函数
2. **哪个会话**——通过 `config` 指定 `session_id`

### 会话隔离

不同的 `session_id` 对应独立的对话历史，互不干扰。

In [110]:
# 另一个用户的会话
config_b = {"configurable": {"session_id": "user_002"}}

response = chain_with_history.invoke(
    {"input": "我叫小红，我是一名设计师"},
    config=config_b,
)
print(f"user_002 助手: {response}")

user_002 助手: 你好小红！很高兴认识你 👋

作为一名设计师，你平时主要做哪方面的设计呢？比如：
- 平面设计（品牌、海报、包装等）
- UI/UX 设计（App、网页界面）
- 插画或视觉艺术
- 室内设计
- 工业设计
- 还是其他领域？

有什么我可以帮你的吗？比如灵感参考、设计资源推荐，或者聊聊设计趋势都可以 😊


In [111]:
# 回到 user_001 的会话——它还记得小明
response = chain_with_history.invoke(
    {"input": "我今年多大？"},
    config={"configurable": {"session_id": "user_001"}},
)
print(f"user_001 助手: {response}")

user_001 助手: 你今年**25岁**。


In [112]:
# user_002 不知道 user_001 的信息
response = chain_with_history.invoke(
    {"input": "我叫什么？我的职业是什么？"},
    config=config_b,
)
print(f"user_002 助手: {response}")

user_002 助手: 你叫**小红**，职业是**设计师** 😊


每个 `session_id` 维护独立的对话历史，这就是多用户/多会话场景的基础。

### RunnableWithMessageHistory 关键参数

| 参数 | 说明 |
|------|------|
| 第 1 个参数 | 被包装的 Runnable（链） |
| 第 2 个参数 | 获取历史的函数，签名：`(session_id) -> BaseChatMessageHistory` |
| `input_messages_key` | 输入字典中用户消息对应的 key |
| `history_messages_key` | 模板中 `MessagesPlaceholder` 的变量名 |
| `output_messages_key` | 输出中 AI 回复对应的 key（链输出为字符串时可不设） |

### 常见问题

- **报错 `session_id` 找不到**：确保 invoke 时传了 `config={"configurable": {"session_id": "xxx"}}`
- **历史没生效**：检查 `history_messages_key` 是否和模板中 `MessagesPlaceholder` 的名字一致
- **内存存储重启丢失**：默认的 `ChatMessageHistory` 存在内存中，进程结束就没了。生产环境用 Redis、MongoDB 等持久化后端

## 5. trim_messages（窗口记忆）

随着对话变长，消息列表会越来越大。模型有上下文窗口限制（比如 8K、32K tokens），超了就会报错或被截断。

`trim_messages` 可以只保留最近的 N 条消息或 N 个 token，丢弃早期内容。

In [113]:
from langchain_core.messages import trim_messages, SystemMessage

# 模拟一段较长的对话
messages = [
    SystemMessage(content="你是一个数学助手"),
    HumanMessage(content="1+1 等于几？"),
    AIMessage(content="1+1=2"),
    HumanMessage(content="2+2 等于几？"),
    AIMessage(content="2+2=4"),
    HumanMessage(content="3+3 等于几？"),
    AIMessage(content="3+3=6"),
    HumanMessage(content="10+10 等于几？"),
    AIMessage(content="10+10=20"),
    HumanMessage(content="还记得第一个问题吗？"),
]

print(f"原始消息数: {len(messages)}")

原始消息数: 10


In [115]:
# 许多非 OpenAI 模型没有实现 get_num_tokens_from_messages()，
# 所以我们用 tiktoken（langchain-openai 已自带）自定义一个 token 计数函数
import tiktoken

_encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(messages):
    """用 tiktoken 估算消息列表的 token 总数"""
    return sum(len(_encoding.encode(msg.content)) for msg in messages)

# 只保留最近的消息，控制在 60 个 token 以内
trimmed = trim_messages(
    messages,
    max_tokens=60,
    token_counter=count_tokens,  # 自定义 token 计数函数
    strategy="last",             # 保留最后的消息
    include_system=True,         # 始终保留 system 消息
    allow_partial=False,         # 不截断单条消息
)

print(f"裁剪后消息数: {len(trimmed)}")
print()
for msg in trimmed:
    print(f"{msg.type}: {msg.content}")

裁剪后消息数: 7

system: 你是一个数学助手
ai: 2+2=4
human: 3+3 等于几？
ai: 3+3=6
human: 10+10 等于几？
ai: 10+10=20
human: 还记得第一个问题吗？


### 刚才发生了什么？

`trim_messages` 从消息列表的末尾开始保留，直到 token 总数接近 `max_tokens` 为止。`include_system=True` 保证 system 消息始终保留（它定义了模型行为）。

早期的对话被丢弃了——这就是“窗口记忆”：只记最近的内容。

> **关于 `token_counter`**：`trim_messages` 的 `token_counter` 参数可以传 LLM 实例（会调用模型自带的 tokenizer），也可以传一个自定义函数。由于许多非 OpenAI 模型没有实现 `get_num_tokens_from_messages()`，这里我们用 `tiktoken` 的 `cl100k_base` 编码来估算 token 数。精确度对于窗口裁剪来说足够了。

### trim_messages 关键参数

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `messages` | 要裁剪的消息列表 | 必填 |
| `max_tokens` | token 上限 | 必填 |
| `token_counter` | token 计数器，可传 LLM 实例或函数 | 必填 |
| `strategy` | `"last"`（保留最新）或 `"first"`（保留最早） | `"last"` |
| `include_system` | 是否始终保留 system 消息 | `False` |
| `allow_partial` | 是否允许截断单条消息 | `False` |
| `start_on` | 裁剪后第一条消息的类型限制（如 `"human"`） | `None` |

### 在链中集成 trim_messages

把 `trim_messages` 和 `RunnableWithMessageHistory` 结合，实现自动窗口记忆。

In [116]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 定义裁剪函数
def trim_history(messages):
    """将消息列表裁剪到 100 token 以内"""
    return trim_messages(
        messages,
        max_tokens=100,
        token_counter=count_tokens,
        strategy="last",
        include_system=True,
        allow_partial=False,
    )

# 构建带裁剪的链
prompt_with_trim = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手。"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

trimmed_chain = (
    RunnablePassthrough.assign(history=lambda x: trim_history(x["history"]))
    | prompt_with_trim
    | llm
    | StrOutputParser()
)

print("带裁剪的链创建完成")

带裁剪的链创建完成


In [117]:
# 为带裁剪的链包装 RunnableWithMessageHistory
store_trimmed = {}

def get_trimmed_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store_trimmed:
        store_trimmed[session_id] = ChatMessageHistory()
    return store_trimmed[session_id]

chain_with_trim = RunnableWithMessageHistory(
    trimmed_chain,
    get_trimmed_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config_trim = {"configurable": {"session_id": "trim_test"}}

# 连续对话多轮
questions = [
    "我叫小明",
    "1+1 等于几？",
    "2+2 等于几？",
    "3+3 等于几？",
    "10+10 等于几？",
    "100+100 等于几？",
    "我叫什么名字？",
]

for q in questions:
    response = chain_with_trim.invoke({"input": q}, config=config_trim)
    print(f"人类: {q}")
    print(f"助手: {response}")
    print()

人类: 我叫小明
助手: 你好小明！很高兴认识你。有什么我可以帮助你的吗？ 😊

人类: 1+1 等于几？
助手: 1 + 1 = **2**

人类: 2+2 等于几？
助手: 2 + 2 = **4**

人类: 3+3 等于几？
助手: 3 + 3 = **6**

人类: 10+10 等于几？
助手: 10 + 10 = **20**

人类: 100+100 等于几？
助手: 100 + 100 = **200**

人类: 我叫什么名字？
助手: 我不知道您的名字。作为AI助手，我没有记忆功能，不会记住之前的对话内容，也无法知道您的个人信息。

如果您愿意的话，可以告诉我您的名字，我很乐意称呼您！



最后问“我叫什么名字？”时，模型大概率无法回答——因为最早的那轮对话已经被 `trim_messages` 裁掉了。

这就是窗口记忆的代价：**控制了 token 用量，但丢失了早期信息**。接下来看一种更聪明的方式。

## 6. 摘要记忆（长对话压缩）

窗口记忆简单粗暴地丢弃旧消息。更好的方式是：**先总结旧消息，再丢弃**。这样关键信息被压缩保留，而不是直接丢失。

思路：
1. 当对话历史超过阈值时，用 LLM 把旧消息总结成一段摘要
2. 用摘要替换旧消息
3. 保留摘要 + 最近几轮对话

In [118]:
# 定义摘要生成链
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "将以下对话总结为一段简短的摘要，保留关键信息（人名、数字、重要事实）。只输出摘要，不要其他内容。"),
    MessagesPlaceholder("messages"),
])

summary_chain = summary_prompt | llm | StrOutputParser()

print("摘要链创建完成")

摘要链创建完成


In [119]:
# 测试摘要链
test_messages = [
    HumanMessage(content="我叫小明，今年 25 岁"),
    AIMessage(content="你好小明！25 岁正是大好年华。"),
    HumanMessage(content="我在北京做程序员"),
    AIMessage(content="北京的程序员工作机会很多呢！"),
    HumanMessage(content="我最近在学 LangChain"),
    AIMessage(content="LangChain 是构建 LLM 应用的好框架。"),
    HumanMessage(content="我养了一只叫豆豆的猫"),
    AIMessage(content="豆豆这个名字很可爱！"),
]

summary = summary_chain.invoke({"messages": test_messages})
print(f"摘要: {summary}")

摘要: 小明，25岁，北京程序员，正在学习LangChain，养了一只叫豆豆的猫。


In [120]:
def summarize_and_trim(messages, max_message_count=6):
    """
    当消息数超过阈值时：
    1. 把旧消息总结成摘要
    2. 用 SystemMessage 形式保留摘要
    3. 只保留摘要 + 最近的消息
    """
    if len(messages) <= max_message_count:
        return messages

    # 分割：旧消息 vs 最近消息
    old_messages = messages[:-max_message_count]
    recent_messages = messages[-max_message_count:]

    # 生成摘要
    summary = summary_chain.invoke({"messages": old_messages})

    # 把摘要作为 SystemMessage 插在最前面
    summary_message = SystemMessage(content=f"之前的对话摘要：{summary}")

    return [summary_message] + recent_messages

print("summarize_and_trim 函数定义完成")

summarize_and_trim 函数定义完成


In [121]:
# 测试摘要压缩
compressed = summarize_and_trim(test_messages, max_message_count=4)

print(f"原始消息数: {len(test_messages)}")
print(f"压缩后消息数: {len(compressed)}")
print()
for msg in compressed:
    print(f"{msg.type}: {msg.content}")

原始消息数: 8
压缩后消息数: 5

system: 之前的对话摘要：小明，25岁，在北京从事程序员工作。
human: 我最近在学 LangChain
ai: LangChain 是构建 LLM 应用的好框架。
human: 我养了一只叫豆豆的猫
ai: 豆豆这个名字很可爱！


### 刚才发生了什么？

8 条消息被压缩成了 5 条（1 条摘要 + 最近 4 条原始消息）。摘要保留了关键信息（名字、年龄、城市、职业、学习内容），而原始的寒暄被丢弃了。

和窗口记忆的对比：

| 方式 | 做法 | 丢失信息 |
|------|------|----------|
| 窗口记忆 | 直接丢弃旧消息 | 早期信息全部丢失 |
| 摘要记忆 | 先总结再丢弃 | 只丢失细节，关键信息被压缩保留 |

代价是每次压缩需要一次额外的 LLM 调用。

In [122]:
# 在链中集成摘要记忆
summary_aware_chain = (
    RunnablePassthrough.assign(
        history=lambda x: summarize_and_trim(x["history"], max_message_count=6)
    )
    | prompt_with_trim
    | llm
    | StrOutputParser()
)

store_summary = {}

def get_summary_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store_summary:
        store_summary[session_id] = ChatMessageHistory()
    return store_summary[session_id]

chain_with_summary = RunnableWithMessageHistory(
    summary_aware_chain,
    get_summary_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config_summary = {"configurable": {"session_id": "summary_test"}}

# 模拟多轮对话
conversations = [
    "我叫张三，今年 30 岁，住在上海",
    "我是一名数据工程师",
    "我最喜欢的编程语言是 Python",
    "我养了两只狗，一只叫旺财，一只叫来福",
    "我最近在学习 LangChain 框架",
    "你还记得我的名字和职业吗？",
]

for q in conversations:
    response = chain_with_summary.invoke({"input": q}, config=config_summary)
    print(f"人类: {q}")
    print(f"助手: {response}")
    print()

人类: 我叫张三，今年 30 岁，住在上海
助手: 你好张三！很高兴认识你。30岁正是事业和生活都充满活力的年纪，住在上海这座国际化大都市，生活节奏应该挺快的吧？

有什么我可以帮你的吗？无论是工作、生活还是其他方面的问题，我都很乐意聊聊 😊

人类: 我是一名数据工程师
助手: 你好张三！数据工程师是个很棒的职位，在数据驱动的时代越来越重要。

作为在上海工作的数据工程师，你平时主要涉及哪些技术栈呢？比如：

- **数据管道**：Airflow、Prefect、Dagster？
- **大数据处理**：Spark、Flink、Kafka？
- **云平台**：阿里云、AWS、Azure？
- **数据仓库**：Hive、ClickHouse、Snowflake、StarRocks？

上海的数据岗位机会很多，金融科技、电商、自动驾驶这些领域都挺活跃的。你现在是在哪个行业？工作中有什么想聊的，或者遇到什么技术难题吗？ 😊

人类: 我最喜欢的编程语言是 Python
助手: 好选择！Python 确实是数据工程师的利器，生态太丰富了：

**数据处理**
- pandas / polars — 日常 ETL 必备
- pyspark — 大数据场景
- dask / ray — 分布式计算

**数据管道**
- 你之前提到的 Airflow，用 Python 写 DAG 很顺手
- 新兴的 Prefect、Mage 也是 Python 优先

**其他利器**
- pydantic — 数据校验
- SQLAlchemy — 数据库交互
- great_expectations — 数据质量

有没有特别喜欢的库？或者想吐槽的？😄

比如 pandas 的内存问题，有人切到 polars 或 duckdb，也有人直接上 Spark——你平时怎么处理大规模数据？

人类: 我养了两只狗，一只叫旺财，一只叫来福
助手: 哈哈，好喜庆的名字！🐕🐕

**旺财**和**来福**——一听就是招财进宝、福气满满，很传统的中国好彩头。它们是什么品种呀？性格差别大吗？

比如：
- 旺财更活泼，来福更稳重？
- 还是反过来，来福才是拆家主力？😄

在上海养狗，出门遛弯方便吗？我听说有些小区和公园对宠物挺友好的，但也有些地方限制比较多。

平时加班多不多，有没有时间陪它们？数据工

即使对话轮数较多，摘要记忆也能让模型记住早期的关键信息（名字、职业等），同时控制上下文长度。

## 7. 实战：多轮对话机器人

综合运用本章知识，构建一个支持多会话、带摘要压缩记忆和流式输出的聊天机器人。

In [123]:
# 完整的对话机器人
bot_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好且博学的助手。回答简洁清晰，不超过 3 句话。"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

def compress_history(messages, max_message_count=6):
    """当消息数超过阈值时，把旧消息压缩成摘要"""
    if len(messages) <= max_message_count:
        return messages

    old_messages = messages[:-max_message_count]
    recent_messages = messages[-max_message_count:]

    summary = summary_chain.invoke({"messages": old_messages})
    summary_message = SystemMessage(content=f"之前的对话摘要：{summary}")

    return [summary_message] + recent_messages

bot_chain = (
    RunnablePassthrough.assign(history=lambda x: compress_history(x["history"]))
    | bot_prompt
    | llm
    | StrOutputParser()
)

# 多会话存储
bot_store = {}

def get_bot_history(session_id: str) -> ChatMessageHistory:
    if session_id not in bot_store:
        bot_store[session_id] = ChatMessageHistory()
    return bot_store[session_id]

chatbot = RunnableWithMessageHistory(
    bot_chain,
    get_bot_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("聊天机器人初始化完成")

聊天机器人初始化完成


In [124]:
def chat(message: str, session_id: str = "default"):
    """便捷聊天函数，支持流式输出"""
    config = {"configurable": {"session_id": session_id}}
    print(f"你: {message}")
    print("助手: ", end="", flush=True)
    for chunk in chatbot.stream({"input": message}, config=config):
        print(chunk, end="", flush=True)
    print("")

In [125]:
# 会话 A
chat("你好！我叫小明，我是一名后端工程师", session_id="session_a")
chat("我最近在学 LangChain，你觉得怎么样？", session_id="session_a")
chat("结合我的职业背景，你觉得我应该重点学哪些模块？", session_id="session_a")

你: 你好！我叫小明，我是一名后端工程师
助手: 你好小明！欢迎，后端工程师。有什么我可以帮你的吗？
你: 我最近在学 LangChain，你觉得怎么样？
助手: LangChain 很实用！它把 LLM 与外部工具、数据源、记忆等模块化整合，能大幅降低开发复杂 AI 应用的门槛。不过生态迭代快，建议边学边跟官方文档，避免踩过期坑。
你: 结合我的职业背景，你觉得我应该重点学哪些模块？
助手: 作为后端工程师，建议优先掌握：

1. **Chains 与 LCEL** — 理解流程编排和异步管道设计
2. **Retrieval** — RAG 架构是落地重点，熟悉向量存储集成
3. **Agents & Tools** — 设计可扩展的工具注册和权限控制
4. **Memory** — 会话状态管理，对接你的存储系统

这些直接映射到后端常见的编排、缓存、接口设计问题。


In [126]:
# 会话 B —— 完全独立
chat("你好！我是一名高中生，想了解人工智能", session_id="session_b")
chat("有什么适合初学者的入门方向吗？", session_id="session_b")

你: 你好！我是一名高中生，想了解人工智能
助手: 你好！人工智能是让计算机模拟人类智能的技术，包括学习、推理和感知等能力。作为高中生，你可以从Python编程和机器学习基础开始学起，网上有很多免费资源比如Coursera和B站教程。有什么具体方向想深入了解吗？
你: 有什么适合初学者的入门方向吗？
助手: 推荐三个方向：**机器学习基础**（理解算法原理）、**Python编程**（最主流的语言）、**计算机视觉或自然语言处理**（应用最有趣的领域）。可以从Kaggle的入门竞赛或吴恩达的免费课程开始，边做项目边学效果最好。


In [127]:
# 回到会话 A —— 它还记得小明的信息
chat("帮我总结一下我们刚才聊了什么", session_id="session_a")

你: 帮我总结一下我们刚才聊了什么
助手: 我们聊了三件事：
1. 你自我介绍是后端工程师小明
2. 我评价了 LangChain 的实用性和学习建议
3. 针对你的背景，我推荐了 Chains/LCEL、Retrieval、Agents/Tools、Memory 四个重点模块


这个聊天机器人具备：
- 多会话隔离（`session_id`）
- 摘要压缩记忆（对话超过阈值时自动压缩旧消息，保留关键信息）
- 流式输出（`stream` 逐字显示）
- 干净的接口（一个 `chat()` 函数搞定）

## 8. 总结

本章学习了：

- LLM 本身无状态，“记忆”靠我们管理消息列表
- 手动管理可行但繁琐
- `ChatMessageHistory` 提供统一的消息存储接口
- `RunnableWithMessageHistory` 自动加载、注入、保存历史
- `trim_messages` 通过窗口裁剪控制上下文长度
- 摘要记忆在丢弃旧消息前先压缩关键信息

### 速查表

| 方式 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 手动消息列表 | 完全控制 | 繁琐、易出错 | 简单脚本 |
| `ChatMessageHistory` | 接口统一 | 仅内存存储 | 开发调试 |
| `RunnableWithMessageHistory` | 全自动管理 | 需理解 session 机制 | 生产应用 |
| `trim_messages` | 控制 token 用量 | 丢失早期信息 | 长对话 |
| 摘要记忆 | 压缩保留关键信息 | 额外 LLM 调用开销 | 超长对话 |

### 记忆方案选择建议



## 下一章

**第五章：Retrieval / RAG（文档加载、向量存储、检索）** —— 目前为止模型只能回答通用知识，下一章将学习如何让模型基于你自己的文档来回答问题。我们将从文档加载、文本切分、向量化、存储到检索，完整走一遍 RAG 流程。